[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/15_Export.ipynb)

# 导出模块 —— 综合指南

## 概述

本 Notebook 是 Semantica **导出模块(Export Module)**的综合指南,该模块支持把知识图谱、实体、关系和数据导出为多种格式。该模块支持 **8 种导出格式**(RDF、JSON、CSV、Graph、YAML、OWL、Vector、LPG),并具备报告生成能力。

**文档**:[API 参考](https://semantica.readthedocs.io/reference/export/)

### 导出模块架构

导出模块由以下部分组成:

#### **核心导出器类**(9 个类)
- `RDFExporter` - RDF 格式导出(Turtle、RDF/XML、JSON-LD、N-Triples、N3)
- `JSONExporter` - JSON 和 JSON-LD 格式导出
- `CSVExporter` - 表格数据的 CSV 格式导出
- `GraphExporter` - 图格式导出(GraphML、GEXF、DOT)
- `SemanticNetworkYAMLExporter` - 语义网络 YAML 导出
- `YAMLSchemaExporter` - 模式(Schema)YAML 导出
- `OWLExporter` - 本体的 OWL 格式导出
- `VectorExporter` - 面向向量库的向量嵌入导出
- `LPGExporter` - 面向 Neo4j、Memgraph 等数据库的 LPG 格式导出

#### **辅助类**(4 个类)
- `RDFSerializer` - 用于格式转换的 RDF 序列化引擎
- `RDFValidator` - 用于语法检查的 RDF 校验引擎
- `NamespaceManager` - RDF 命名空间管理与冲突解决
- `ReportGenerator` - 报告生成(HTML、Markdown、JSON、文本)

#### **注册表与配置**(2 个类 + 2 个实例)
- `MethodRegistry` - 自定义导出方法的注册表
- `method_registry` - 全局注册表实例
- `ExportConfig` - 导出模块的配置管理器
- `export_config` - 全局配置实例

### 学习目标

学完本 Notebook 后,你将能够:
- 将知识图谱导出为所有受支持的格式(RDF、JSON、CSV、Graph、YAML、OWL、Vector、LPG)
- 直接使用导出器类进行细粒度控制
- 生成多种格式的专业报告
- 注册并使用自定义导出方法
- 通过环境变量或配置文件配置导出设置
- 使用 RDF 序列化、校验和命名空间管理

## 安装

从 PyPI 安装 Semantica:

```bash
pip install semantica
# 或安装全部可选依赖:
pip install semantica[all]
```

---

## 步骤 1:JSON 导出

使用 `JSONExporter` 类将知识图谱导出为 JSON 格式。

**JSONExporter 方法:**
- `export()` - 将任意数据导出为 JSON
- `export_knowledge_graph()` - 将知识图谱导出为 JSON/JSON-LD
- `export_entities()` - 将实体导出为 JSON
- `export_relationships()` - 将关系导出为 JSON

In [ ]:
!pip install semantica

In [ ]:
from semantica.export import JSONExporter
from semantica.kg import GraphBuilder

# Create exporter and builder
json_exporter = JSONExporter()
builder = GraphBuilder()

# Create sample knowledge graph
entities = [{"id": "e1", "type": "Organization", "name": "Apple Inc.", "properties": {}}]
relationships = []

kg = builder.build(entities + relationships)

# Export to JSON
json_exporter.export_knowledge_graph(kg, "output.json")


## 步骤 2:CSV 导出

使用 `CSVExporter` 类将实体和关系导出为 CSV 格式。

**CSVExporter 方法:**
- `export()` - 将知识图谱导出为 CSV
- `export_entities()` - 将实体导出为 CSV 文件
- `export_relationships()` - 将关系导出为 CSV 文件
- `export_knowledge_graph()` - 将完整的知识图谱导出为 CSV

In [ ]:
from semantica.export import CSVExporter

# Create CSV exporter
csv_exporter = CSVExporter()

# Export entities to CSV
csv_exporter.export_entities(entities, "entities.csv")


## 步骤 3:RDF 导出

使用 `RDFExporter` 类将知识图谱导出为 RDF 格式。

**RDFExporter 方法:**
- `export()` - 导出为 RDF(支持多种格式)
- `export_knowledge_graph()` - 将知识图谱导出为 RDF
- `export_entities()` - 将实体导出为 RDF
- `export_relationships()` - 将关系导出为 RDF

**支持的 RDF 格式:**
- `turtle` - Turtle 格式(人类可读)
- `rdfxml` - RDF/XML 格式
- `jsonld` - JSON-LD 格式
- `ntriples` - N-Triples 格式
- `n3` - N3 格式

**其他 RDF 类:**
- `RDFSerializer` - 在不同格式之间序列化 RDF 数据
- `RDFValidator` - 校验 RDF 语法和一致性
- `NamespaceManager` - 管理 RDF 命名空间

In [ ]:
from semantica.export import RDFExporter

# Create RDF exporter
rdf_exporter = RDFExporter()

# Export to RDF format (Turtle by default)
rdf_exporter.export(kg, "output.ttl", format="turtle")

In [ ]:
# TTL alias: format="ttl" is equivalent to format="turtle"
rdf_data = {
    "entities": [
        {"id": "e1", "text": "Apple Inc.", "type": "ORG", "confidence": 0.95},
        {"id": "e2", "text": "Steve Jobs", "type": "PERSON", "confidence": 0.97},
    ],
    "relationships": [
        {"source_id": "e2", "target_id": "e1", "type": "founded_by", "confidence": 0.91},
    ],
}

rdf_exporter.export(rdf_data, "output.ttl", format="ttl")

result = rdf_exporter.validate_rdf(rdf_data)
print(f"Valid: {result['overall_valid']}")

## 步骤 4:图导出

使用 `GraphExporter` 类导出为图格式,供可视化工具使用。

**GraphExporter 方法:**
- `export()` - 导出为图格式
- `export_knowledge_graph()` - 将知识图谱导出为图格式

**支持的图格式:**
- `graphml` - GraphML 格式(用于 Cytoscape、yEd 等)
- `gexf` - GEXF 格式(用于 Gephi)
- `dot` - Graphviz DOT 格式

In [ ]:
from semantica.export import GraphExporter

# Create graph exporter
graph_exporter = GraphExporter()

# Export to GraphML format
graph_exporter.export_knowledge_graph(kg, "output.graphml", format="graphml")


## 步骤 5:OWL 导出

使用 `OWLExporter` 类将本体导出为 OWL 格式。

**OWLExporter 方法:**
- `export()` - 将本体导出为 OWL
- `export_ontology()` - 导出完整的本体
- `export_classes()` - 仅导出类定义
- `export_properties()` - 仅导出属性定义

**支持的 OWL 格式:**
- `owl-xml` - OWL/XML 格式(默认)
- `turtle` - Turtle 格式的 OWL

**注意:**OWLExporter 期望的是本体结构(包含类、属性),而不是知识图谱。请先使用 `OntologyGenerator` 把知识图谱转换为本体。

In [ ]:
from semantica.export import OWLExporter
from semantica.ontology import OntologyGenerator

# Create OWL exporter and ontology generator
owl_exporter = OWLExporter()
generator = OntologyGenerator()

# Generate ontology from entities and relationships
# Note: Pass data as a dictionary to generate_ontology
ontology = generator.generate_ontology({
    "entities": entities,
    "relationships": relationships
})

# Export ontology to OWL
owl_exporter.export(ontology, "output.owl")

## 步骤 6:其他导出格式

### YAML 导出

使用 YAML 导出器导出为 YAML 格式:

**YAML 导出器类:**
- `SemanticNetworkYAMLExporter` - 将语义网络导出为 YAML
- `YAMLSchemaExporter` - 将本体模式导出为 YAML

### 向量导出

使用 `VectorExporter` 导出向量嵌入:

**VectorExporter 方法:**
- `export()` - 将向量导出为多种格式

**支持的向量格式:**
- `json` - JSON 格式
- `numpy` - NumPy 格式
- `binary` - 二进制格式
- `faiss` - FAISS 格式

### LPG 导出

使用 `LPGExporter` 导出为标签属性图(Labeled Property Graph)格式:

**LPGExporter 方法:**
- `export()` - 导出为 LPG 格式
- `export_knowledge_graph()` - 将知识图谱导出为 LPG

**支持的 LPG 格式:**
- `cypher` - Cypher 查询格式(用于 Neo4j、Memgraph)
- `lpg` - 标签属性图格式

### 报告生成

使用 `ReportGenerator` 生成专业报告:

**ReportGenerator 方法:**
- `generate_report()` - 生成多种格式的报告

**支持的报告格式:**
- `html` - HTML 报告
- `markdown` - Markdown 报告
- `json` - JSON 报告
- `text` - 纯文本报告


## 总结

你已经学会如何使用 Semantica 的导出模块导出数据:

### **核心导出器类:**
- **JSONExporter**:导出为 JSON/JSON-LD 格式
- **CSVExporter**:导出为 CSV 格式
- **RDFExporter**:导出为 RDF 格式(Turtle、RDF/XML、JSON-LD、N-Triples、N3)
- **GraphExporter**:导出为图格式(GraphML、GEXF、DOT)
- **OWLExporter**:将本体导出为 OWL
- **VectorExporter**:将向量导出为多种格式
- **LPGExporter**:导出为标签属性图格式
- **SemanticNetworkYAMLExporter**:将语义网络导出为 YAML
- **YAMLSchemaExporter**:将模式导出为 YAML

### **辅助类:**
- **RDFSerializer**:RDF 格式转换
- **RDFValidator**:RDF 校验
- **NamespaceManager**:RDF 命名空间管理
- **ReportGenerator**:生成专业报告

### **注册表与配置:**
- `MethodRegistry`:注册自定义导出方法
- `ExportConfig`:配置导出设置
- `method_registry`:用于访问已注册方法的全局注册表实例
- `export_config`:用于导出设置的全局配置实例

**下一步:**
- 在[多格式导出 Notebook](../advanced/05_Multi_Format_Export.ipynb) 中学习高级导出技术
- 在 Visualization Notebook 中学习如何可视化数据